# 01 · LLM Answers

**Goal:** ask a large language model (LLM) to match every VAL entry, and save its answers, so `02_evaluation` can compare it with the trained model. The LLM is only a comparison: matching itself doesn't use one.

The LLM gets most of what the model gets: the entry as written, the firm's matter list with types and dates, the timekeeper's most-billed matters, and 3 past entries for each matter. It doesn't get codenames, client aliases, or every past entry, which the model uses (`02_evaluation`, section 14, has what this means for the comparison). It answers through a tool call with fixed fields: the matter (or NONE for a new client or matter), a second matter if the entry covers two, a shortlist, how sure it is from 0 to 100, and a one-line reason.

**This notebook is the only one that pays for calls** (Amazon Bedrock, about 630 calls). Each answer is saved in `data/llm_answers/` as it comes back, so a rerun only asks about entries without one. Needs AWS credentials and `00_build`'s `00_entries.parquet`.

## 1. Setup

In [1]:
import json
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import boto3
import pandas as pd
from botocore.config import Config

sys.path.insert(0, "../src")
sys.path.insert(0, "../../extraction/src")
from pipeline import HISTORY_COLS, MATTER_COLS

In [2]:
DATA_PATH = Path("../data/synthetic_time_entries_v3.xlsx")
RESULTS_DIR = Path("../results")

REGION = "us-west-2"
MODEL_ID = "us.amazon.nova-pro-v1:0"   # any Bedrock model with tool use; a stronger one gives the LLM its best case
ANSWERS_DIR = Path("../data/llm_answers") / MODEL_ID.replace(":", "_")
WORKERS = 4
SAMPLE = None   # a number asks only the first that many VAL entries, to try the prompt cheaply first
REPEAT_SAMPLE = 30   # entries asked a second time, to see if the answer stays the same

PAST_PER_MATTER = 3   # past entries shown for each matter
HISTORY_TOP = 8   # the timekeeper's most-billed matters shown with each entry
HISTORY_LABEL = "April 1 to June 30, 2026"

## 2. Load data

- **VAL entries** with email fields filled in, from `00_build`.
- **Matter Master and timekeeper history**, the same tables the model reads.
- **Past entries for each matter:** TRAIN entries with one known matter, the same billing history the past-narrative clue uses.

In [3]:
sheets = pd.read_excel(DATA_PATH, sheet_name=None, dtype=str)
raw = sheets["Time Entries"]
matters = sheets["Matter Master"][list(MATTER_COLS)].rename(columns=MATTER_COLS)
history = sheets["TK Matter History"][list(HISTORY_COLS)].rename(columns=HISTORY_COLS)
history["hours"] = pd.to_numeric(history["hours"], errors="coerce")
entries = pd.read_parquet(RESULTS_DIR / "00_entries.parquet")
key = raw[["Time ID", "KEY: Split", "KEY: True CMID", "KEY: Secondary CMID"]].set_axis(["entry_id", "split", "true_cmid", "second_cmid"], axis=1)
entries = entries.merge(key, on="entry_id")

val = entries[entries["split"] == "VAL"].reset_index(drop=True)
if SAMPLE:
    val = val.head(SAMPLE)
past = entries[(entries["split"] == "TRAIN") & (entries["true_cmid"] != "NONE") & entries["second_cmid"].isna()]
examples = past.groupby("true_cmid")["narrative"].apply(lambda s: list(s.head(PAST_PER_MATTER)))

assert val["entry_id"].is_unique and not set(val["entry_id"]) & set(past["entry_id"])
assert set(examples.index) <= set(matters["cmid"])
print(f"VAL entries: {len(val)} | matters: {len(matters)} | matters with past entries: {len(examples)}")

VAL entries: 601 | matters: 103 | matters with past entries: 86


## 3. The prompt

- **System prompt:** the instructions, the matter list with types and open and close dates, and a few past entries for each matter. It's the same for every entry, so Bedrock's prompt cache can reuse it.
- **One message per entry:** the typed fields as written (blank ones say so), the narrative, the work date, and the timekeeper's most-billed matters.
- **The tool call** forces the answer into fixed fields, so every answer is scored the same way.

In [4]:
INSTRUCTIONS = """You assign law firm time entries to the right client matter.

Each entry was written by a timekeeper. The typed CMID (client matter number), client, and matter can be missing, misspelled, or wrong. The narrative says what work was done.

Use the matter list, the timekeeper's billing history, and the past entries for each matter:
- Pick the matter the work belongs to. A typed CMID can be wrong: if the narrative clearly points to another matter, pick that one.
- Time can't go to a matter before it opened or after it closed.
- If the entry is for a client or matter that isn't on the list, answer NONE.
- If the entry also covers work on a second matter, give it as second_cmid.
- confidence: 0 to 100, how likely it is that the first matter is right and the entry covers only that matter. Give 90 or more only when the match is safe to accept with no one checking."""

TOOL = {"toolSpec": {"name": "match_entry", "description": "Record the matter for this time entry.", "inputSchema": {"json": {
    "type": "object",
    "properties": {
        "cmid": {"type": "string", "description": "The matter's CMID from the list, or NONE"},
        "second_cmid": {"type": "string", "description": "A second matter's CMID if the entry covers two, else empty"},
        "shortlist": {"type": "array", "items": {"type": "string"}, "description": "Up to 3 likely CMIDs, most likely first"},
        "confidence": {"type": "integer", "description": "0 to 100"},
        "reason": {"type": "string", "description": "One short sentence"},
    },
    "required": ["cmid", "second_cmid", "shortlist", "confidence", "reason"]}}}}


def blank(value):
    return value is None or pd.isna(value) or str(value).strip() == ""


def matter_line(m):
    closed = "still open" if blank(m["close_date"]) else str(m["close_date"])[:10]
    return f"{m['cmid']} | {m['client']} | {m['matter']} | {m['matter_type']} | opened {str(m['open_date'])[:10]} | closed {closed}"


def entry_text(row):
    billed = history[history["tk_id"] == row["tk_id"]].sort_values("hours", ascending=False).head(HISTORY_TOP)
    hours = ", ".join(f"{c} ({h:.1f} h)" for c, h in zip(billed["cmid"], billed["hours"])) or "none"
    typed = {field: "(blank)" if blank(row[field]) else str(row[field]) for field in ["cmid", "client", "matter"]}
    return (f"Timekeeper: {row['tk_id']}\nWork date: {pd.Timestamp(row['work_date']).date()}\n"
            f"Typed CMID: {typed['cmid']}\nTyped client: {typed['client']}\nTyped matter: {typed['matter']}\n"
            f"Narrative: {row['narrative']}\n"
            f"This timekeeper's hours by matter, {HISTORY_LABEL}: {hours}")


matter_list = "\n".join(matter_line(m) for m in matters.to_dict("records"))
past_list = "\n".join(f"{cmid}: " + " / ".join(texts) for cmid, texts in examples.items())
SYSTEM = f"{INSTRUCTIONS}\n\nMatter list (CMID | client | matter | type | opened | closed):\n{matter_list}\n\nPast entries by matter:\n{past_list}"

print(f"System prompt: about {len(SYSTEM.split()):,} words\n")
print(entry_text(val.iloc[0]))

System prompt: about 4,876 words

Timekeeper: TK104
Work date: 2026-07-06
Typed CMID: (blank)
Typed client: Meridian Therapeutics
Typed matter: sec
Narrative: Dft proxy statement re MD&A; flag open issues.
This timekeeper's hours by matter, April 1 to June 30, 2026: 6103.004 (125.4 h), 5602.001 (34.7 h), 2960.003 (12.9 h), 1111.001 (6.3 h), 3455.001 (5.0 h), 9140.001 (3.0 h), 8012.001 (1.8 h), 6103.002 (1.4 h)


## 4. Ask

A broken tool call is asked again up to 3 times. A call that still fails isn't saved, so a rerun asks again, and `02_evaluation` counts the entry as sent to a person with no matter.

In [5]:
runtime = boto3.client("bedrock-runtime", region_name=REGION, config=Config(retries={"max_attempts": 8, "mode": "adaptive"}))


def ask(row, folder=ANSWERS_DIR, tries=3):
    path = folder / f"{row['entry_id']}.json"
    if path.exists():
        return json.loads(path.read_text())
    for attempt in range(tries):
        try:
            start = time.perf_counter()
            response = runtime.converse(
                modelId=MODEL_ID,
                system=[{"text": SYSTEM}, {"cachePoint": {"type": "default"}}],
                messages=[{"role": "user", "content": [{"text": entry_text(row)}]}],
                toolConfig={"tools": [TOOL], "toolChoice": {"tool": {"name": "match_entry"}}},
                inferenceConfig={"temperature": 0, "maxTokens": 400})
            break
        except Exception as error:
            if "ModelErrorException" not in type(error).__name__ + str(error) or attempt == tries - 1:
                return {"entry_id": row["entry_id"], "error": f"{type(error).__name__}: {error}"}
    found = [b["toolUse"]["input"] for b in response["output"]["message"]["content"] if "toolUse" in b]
    answer = {"entry_id": row["entry_id"], **(found[0] if found else {}), "usage": response["usage"],
              "latency_s": response["metrics"]["latencyMs"] / 1000, "seconds": time.perf_counter() - start}
    if found:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(answer, indent=1))
    return answer


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    answers = list(pool.map(ask, val.to_dict("records")))
failed = [a for a in answers if "error" in a or "cmid" not in a]
print(f"{len(answers)} answers in {time.perf_counter() - start:.0f} s | failed: {len(failed)}")
pd.DataFrame(failed[:5])

601 answers in 664 s | failed: 0


""


## 5. The same entries again

The first `REPEAT_SAMPLE` entries are asked a second time and saved apart (`data/llm_answers/.../repeat/`). At temperature 0 the answers should mostly match. `02_evaluation` compares the two.

In [6]:
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    again = list(pool.map(lambda row: ask(row, ANSWERS_DIR / "repeat"), val.head(REPEAT_SAMPLE).to_dict("records")))

usage = pd.DataFrame([a.get("usage", {}) for a in answers]).fillna(0)
print(f"Saved: {len(list(ANSWERS_DIR.glob('*.json')))} answers, {len(list((ANSWERS_DIR / 'repeat').glob('*.json')))} repeats")
usage.mean().round(0).rename("tokens per call").to_frame()

Saved: 601 answers, 30 repeats


,tokens per call
inputTokens,202.0
outputTokens,86.0
totalTokens,9920.0
cacheReadInputTokens,9584.0
cacheWriteInputTokens,48.0
